# WiDS Global Datathon 2026 — Hierarchical Hazard Superlearner


In [1]:
# WiDS Global Datathon 2026 — Hierarchical Hazard Superlearner

import os
import sys
import gc
import math
import glob
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.special import expit, logit
from scipy.stats import norm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import NearestNeighbors

import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor

SEED = 2026
rng = np.random.default_rng(SEED)
HORIZONS = [12, 24, 48, 72]
INTERVALS = [(0, 12), (12, 24), (24, 48), (48, 72)]
REQUIRED_SUB_COLS = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]

def find_competition_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/wids-global-datathon-2026",
        "/kaggle/input/anonymous-contest-2",
        ".",
    ]
    for p in candidates:
        if Path(p, "train.csv").exists() and Path(p, "test.csv").exists():
            return Path(p)
    for p in Path("/kaggle/input").glob("**/train.csv"):
        if p.parent.joinpath("test.csv").exists():
            return p.parent
    raise FileNotFoundError("Could not locate train.csv/test.csv")

DATA_DIR = find_competition_dir()
print("DATA_DIR =", DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv") if (DATA_DIR / "sample_submission.csv").exists() else None

assert "event_id" in train.columns and "event_id" in test.columns
assert "time_to_hit_hours" in train.columns and "event" in train.columns

def clip01(x):
    return np.clip(np.asarray(x, dtype=float), 1e-6, 1 - 1e-6)

def add_engineered_features(df):
    df = df.copy()

    if "dist_min_ci_0_5h" in df.columns:
        df["dist_km"] = df["dist_min_ci_0_5h"] / 1000.0
        df["log_dist"] = np.log1p(np.clip(df["dist_km"], 0, None))
        df["inv_dist"] = 1.0 / (0.25 + np.clip(df["dist_km"], 0, None))
        df["near_2km"] = (df["dist_km"] < 2).astype(int)
        df["near_5km"] = (df["dist_km"] < 5).astype(int)
        df["near_10km"] = (df["dist_km"] < 10).astype(int)
        df["mid_5_15km"] = ((df["dist_km"] >= 5) & (df["dist_km"] < 15)).astype(int)
        df["far_15km"] = (df["dist_km"] >= 15).astype(int)
        df["far_25km"] = (df["dist_km"] >= 25).astype(int)
        bins = np.array([2, 5, 10, 15, 25, 40], dtype=float)
        dist = df["dist_km"].to_numpy()
        df["zone_id"] = np.digitize(dist, bins, right=False).astype(int)

    if "closing_speed_m_per_h" in df.columns:
        df["closing_pos"] = np.clip(df["closing_speed_m_per_h"], 0, None)
        df["closing_neg"] = np.clip(-df["closing_speed_m_per_h"], 0, None)
        df["closing_abs_log"] = np.log1p(np.abs(df["closing_speed_m_per_h"]))
    else:
        df["closing_pos"] = 0.0
        df["closing_neg"] = 0.0
        df["closing_abs_log"] = 0.0

    if "along_track_speed" in df.columns:
        df["along_pos"] = np.clip(df["along_track_speed"], 0, None)
        df["along_neg"] = np.clip(-df["along_track_speed"], 0, None)
        df["along_abs_log"] = np.log1p(np.abs(df["along_track_speed"]))
    else:
        df["along_pos"] = 0.0
        df["along_neg"] = 0.0
        df["along_abs_log"] = 0.0

    if "alignment_abs" in df.columns:
        df["align_pos_close"] = df["alignment_abs"] * df["closing_pos"]
        df["align_pos_along"] = df["alignment_abs"] * df["along_pos"]
        df["align_sq"] = np.square(df["alignment_abs"])
    else:
        df["align_pos_close"] = 0.0
        df["align_pos_along"] = 0.0
        df["align_sq"] = 0.0

    if "projected_advance_m" in df.columns and "dist_min_ci_0_5h" in df.columns:
        df["advance_ratio"] = df["projected_advance_m"] / (100.0 + np.clip(df["dist_min_ci_0_5h"], 0, None))
        df["advance_pos"] = np.clip(df["projected_advance_m"], 0, None)
    else:
        df["advance_ratio"] = 0.0
        df["advance_pos"] = 0.0

    if "dist_min_ci_0_5h" in df.columns:
        df["ttc_close_h"] = np.clip(df["dist_min_ci_0_5h"] / np.clip(df["closing_pos"], 1.0, None), 0, 500)
        df["ttc_along_h"] = np.clip(df["dist_min_ci_0_5h"] / np.clip(df["along_pos"], 1.0, None), 0, 500)
        df["ttc_inv"] = 1.0 / (1.0 + df["ttc_close_h"])
        df["eta_12"] = (df["ttc_close_h"] <= 12).astype(int)
        df["eta_24"] = (df["ttc_close_h"] <= 24).astype(int)
        df["eta_48"] = (df["ttc_close_h"] <= 48).astype(int)
    else:
        df["ttc_close_h"] = 500.0
        df["ttc_along_h"] = 500.0
        df["ttc_inv"] = 0.0
        df["eta_12"] = 0
        df["eta_24"] = 0
        df["eta_48"] = 0

    if "dt_first_last_0_5h" in df.columns and "num_perimeters_0_5h" in df.columns:
        df["obs_density"] = df["num_perimeters_0_5h"] / (0.25 + df["dt_first_last_0_5h"])
        df["quality_score"] = (1 - df.get("low_temporal_resolution_0_5h", 0)) * (0.25 + df["dt_first_last_0_5h"])
    else:
        df["obs_density"] = 0.0
        df["quality_score"] = 0.0

    if "area_growth_rate_ha_per_h" in df.columns:
        df["growth_rate_pos"] = np.clip(df["area_growth_rate_ha_per_h"], 0, None)
        df["growth_rate_log"] = np.log1p(np.clip(df["area_growth_rate_ha_per_h"], 0, None))
    else:
        df["growth_rate_pos"] = 0.0
        df["growth_rate_log"] = 0.0

    if "radial_growth_rate_m_per_h" in df.columns:
        df["radial_rate_pos"] = np.clip(df["radial_growth_rate_m_per_h"], 0, None)
        df["radial_rate_log"] = np.log1p(np.clip(df["radial_growth_rate_m_per_h"], 0, None))
    else:
        df["radial_rate_pos"] = 0.0
        df["radial_rate_log"] = 0.0

    if "log1p_area_first" in df.columns:
        df["size_dist_ratio"] = df["log1p_area_first"] / (0.25 + df.get("log_dist", 0))
    else:
        df["size_dist_ratio"] = 0.0

    if "log1p_growth" in df.columns:
        df["growth_dist_ratio"] = df["log1p_growth"] / (0.25 + df.get("log_dist", 0))
    else:
        df["growth_dist_ratio"] = 0.0

    df["threat_proxy"] = (
        1.75 * df.get("inv_dist", 0)
        + 0.70 * df.get("ttc_inv", 0)
        + 0.40 * np.tanh(df.get("align_pos_close", 0) / 2000.0)
        + 0.25 * np.tanh(df.get("growth_rate_log", 0))
        + 0.15 * np.tanh(df.get("advance_ratio", 0))
    )

    cyc = {
        "event_start_hour": 24,
        "event_start_dayofweek": 7,
        "event_start_month": 12,
    }
    for col, period in cyc.items():
        if col in df.columns:
            vals = df[col].astype(float).fillna(0).to_numpy()
            df[f"{col}_sin"] = np.sin(2 * np.pi * vals / period)
            df[f"{col}_cos"] = np.cos(2 * np.pi * vals / period)

    for col in ["spread_bearing_deg"]:
        if col in df.columns:
            vals = np.deg2rad(df[col].astype(float).fillna(0).to_numpy())
            df[f"{col}_sin2"] = np.sin(vals)
            df[f"{col}_cos2"] = np.cos(vals)

    return df

train = add_engineered_features(train)
test = add_engineered_features(test)

target_cols = ["time_to_hit_hours", "event"]
feature_cols = [c for c in train.columns if c not in ["event_id"] + target_cols]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]
feature_cols = [c for c in feature_cols if c in test.columns]
feature_cols = sorted(feature_cols)

core_meta_cols = [
    c for c in [
        "dist_km","log_dist","inv_dist","zone_id","near_2km","near_5km","near_10km","mid_5_15km","far_15km","far_25km",
        "closing_pos","along_pos","alignment_abs","align_sq","align_pos_close","advance_ratio","threat_proxy",
        "ttc_close_h","ttc_inv","growth_rate_log","radial_rate_log","size_dist_ratio","growth_dist_ratio",
        "quality_score","obs_density","low_temporal_resolution_0_5h"
    ] if c in train.columns
]

print("TRAIN =", train.shape, "TEST =", test.shape, "N_FEATURES =", len(feature_cols))

def prepare_xy(train_df, valid_df, test_df, cols):
    med = train_df[cols].median(numeric_only=True)
    Xtr = train_df[cols].fillna(med)
    Xva = valid_df[cols].fillna(med)
    Xte = test_df[cols].fillna(med)
    return Xtr, Xva, Xte

def cumulative_label(df, horizon):
    t = df["time_to_hit_hours"].to_numpy(dtype=float)
    e = df["event"].to_numpy(dtype=int)
    y = ((e == 1) & (t <= horizon)).astype(int)
    mask = (e == 1) | (t >= horizon)
    return y, mask

def interval_label(df, start, end):
    t = df["time_to_hit_hours"].to_numpy(dtype=float)
    e = df["event"].to_numpy(dtype=int)
    y = ((e == 1) & (t > start) & (t <= end)).astype(int)
    at_risk = t > start
    observed_to_end = (e == 1) | (t >= end)
    mask = at_risk & observed_to_end
    return y, mask

def cumulative_to_hazards(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    if p.ndim == 1:
        p = p.reshape(-1, 4)
    h1 = p[:, 0]
    h2 = (p[:, 1] - p[:, 0]) / np.clip(1 - p[:, 0], 1e-6, None)
    h3 = (p[:, 2] - p[:, 1]) / np.clip(1 - p[:, 1], 1e-6, None)
    h4 = (p[:, 3] - p[:, 2]) / np.clip(1 - p[:, 2], 1e-6, None)
    h = np.column_stack([h1, h2, h3, h4])
    return np.clip(h, 1e-6, 1 - 1e-6)

def hazards_to_cumulative(h):
    h = np.clip(np.asarray(h, dtype=float), 1e-6, 1 - 1e-6)
    if h.ndim == 1:
        h = h.reshape(-1, 4)
    s = np.ones(len(h), dtype=float)
    out = []
    for j in range(h.shape[1]):
        s = s * (1 - h[:, j])
        out.append(1 - s)
    return np.column_stack(out)

def monotone_project(p):
    p = np.asarray(p, dtype=float).copy()
    p[:, 0] = np.clip(p[:, 0], 0, 1)
    for j in range(1, p.shape[1]):
        p[:, j] = np.maximum(p[:, j], p[:, j - 1])
        p[:, j] = np.clip(p[:, j], 0, 1)
    return p

def urgency_score_from_cumulative(p):
    h = cumulative_to_hazards(p)
    mids = np.array([6, 18, 36, 60], dtype=float)
    surv_prev = np.ones(len(h))
    exp_t = np.zeros(len(h))
    for j in range(4):
        mass = surv_prev * h[:, j]
        exp_t += mids[j] * mass
        surv_prev *= (1 - h[:, j])
    exp_t += 84.0 * surv_prev
    return -exp_t

def concordance_index_approx(time, event, score):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    score = np.asarray(score, dtype=float)
    num = 0.0
    den = 0.0
    n = len(time)
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j:
                continue
            if time[i] < time[j]:
                den += 1
                if score[i] > score[j]:
                    num += 1
                elif score[i] == score[j]:
                    num += 0.5
    return num / den if den > 0 else 0.5

def weighted_brier_from_probs(df, p):
    out = {}
    for H in [12, 24, 48, 72]:
        y, mask = cumulative_label(df, H)
        if mask.sum() == 0:
            out[H] = np.nan
        else:
            j = HORIZONS.index(H)
            out[H] = float(np.mean((p[mask, j] - y[mask]) ** 2))
    wb = 0.3 * out.get(24, np.nan) + 0.4 * out.get(48, np.nan) + 0.3 * out.get(72, np.nan)
    return out, wb

def hybrid_score_from_probs(df, p):
    p = monotone_project(np.asarray(p, dtype=float))
    score = urgency_score_from_cumulative(p)
    c = concordance_index_approx(df["time_to_hit_hours"].to_numpy(), df["event"].to_numpy(), score)
    brier_by_h, wb = weighted_brier_from_probs(df, p)
    hybrid = 0.3 * c + 0.7 * (1 - wb)
    return hybrid, c, wb, brier_by_h

def zone_codes(df):
    if "zone_id" in df.columns:
        return df["zone_id"].to_numpy(dtype=int)
    dist = df.get("dist_km", pd.Series(np.full(len(df), 999.0))).to_numpy(dtype=float)
    return np.digitize(dist, [2, 5, 10, 15, 25, 40], right=False).astype(int)

def make_stratify_label(df):
    z = zone_codes(df)
    e = df["event"].to_numpy(dtype=int)
    coarse = np.where(z <= 1, 0, np.where(z <= 3, 1, 2))
    return np.array([f"{a}_{b}" for a, b in zip(e, coarse)])

def class_balanced_weights(y, base=None):
    y = np.asarray(y, dtype=int)
    pos = max(int(y.sum()), 1)
    neg = max(len(y) - int(y.sum()), 1)
    w = np.where(y == 1, neg / pos, 1.0).astype(float)
    if base is not None:
        w = w * np.asarray(base, dtype=float)
    return w

def fit_lgb_classifier(X, y, seed=SEED, sample_weight=None):
    if len(np.unique(y)) < 2:
        return None
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=350,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=8,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.75,
        reg_alpha=0.4,
        reg_lambda=2.0,
        random_state=seed,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X, y, sample_weight=sample_weight)
    return model

def fit_cat_classifier(X, y, seed=SEED, sample_weight=None):
    if len(np.unique(y)) < 2:
        return None
    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=350,
        depth=4,
        learning_rate=0.03,
        l2_leaf_reg=6.0,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False
    )
    model.fit(X, y, sample_weight=sample_weight, verbose=False)
    return model

def fit_cat_regressor(X, y, seed=SEED, sample_weight=None):
    if len(y) < 5:
        return None
    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=350,
        depth=4,
        learning_rate=0.03,
        l2_leaf_reg=6.0,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False
    )
    model.fit(X, y, sample_weight=sample_weight, verbose=False)
    return model

def predict_model(model, X):
    if model is None:
        return None
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.predict(X)

def empirical_anchor(train_df, pred_df):
    tr = train_df.copy()
    te = pred_df.copy()

    tr_zone = zone_codes(tr)
    te_zone = zone_codes(te)

    tr_close = (tr.get("closing_pos", pd.Series(np.zeros(len(tr)))) > 0).astype(int).to_numpy()
    te_close = (te.get("closing_pos", pd.Series(np.zeros(len(te)))) > 0).astype(int).to_numpy()

    tr_align = (tr.get("alignment_abs", pd.Series(np.zeros(len(tr)))) > 0.7).astype(int).to_numpy()
    te_align = (te.get("alignment_abs", pd.Series(np.zeros(len(te)))) > 0.7).astype(int).to_numpy()

    tr_key = tr_zone * 4 + tr_close * 2 + tr_align
    te_key = te_zone * 4 + te_close * 2 + te_align

    global_preds = {}
    for H in HORIZONS:
        y, mask = cumulative_label(tr, H)
        global_mean = (y[mask].sum() + 1.5) / (mask.sum() + 3.0)
        global_preds[H] = global_mean

    stats = {}
    for key in np.unique(tr_key):
        idx = tr_key == key
        stats[key] = {}
        for H in HORIZONS:
            y, mask = cumulative_label(tr.loc[idx], H)
            n = int(mask.sum())
            s = int(y[mask].sum()) if n > 0 else 0
            base = global_preds[H]
            alpha = 8.0
            stats[key][H] = (s + alpha * base) / (n + alpha)

    out = np.zeros((len(te), 4), dtype=float)
    for i, key in enumerate(te_key):
        if key in stats:
            out[i] = [stats[key][H] for H in HORIZONS]
        else:
            out[i] = [global_preds[H] for H in HORIZONS]
    return monotone_project(out)

def linear_cumulative_model(train_df, valid_df, test_df, cols):
    Xtr, Xva, Xte = prepare_xy(train_df, valid_df, test_df, cols)
    pred_va = np.zeros((len(valid_df), 4), dtype=float)
    pred_te = np.zeros((len(test_df), 4), dtype=float)

    for j, H in enumerate(HORIZONS):
        y, mask = cumulative_label(train_df, H)
        if H == 72:
            pred_va[:, j] = 1.0
            pred_te[:, j] = 1.0
            continue
        if mask.sum() < 10 or len(np.unique(y[mask])) < 2:
            base = y[mask].mean() if mask.sum() else train_df["event"].mean()
            pred_va[:, j] = base
            pred_te[:, j] = base
            continue
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.35, class_weight="balanced", max_iter=3000, random_state=SEED))
        ])
        sw = class_balanced_weights(y[mask])
        pipe.fit(Xtr.loc[mask], y[mask], clf__sample_weight=sw)
        pred_va[:, j] = pipe.predict_proba(Xva)[:, 1]
        pred_te[:, j] = pipe.predict_proba(Xte)[:, 1]

    return monotone_project(pred_va), monotone_project(pred_te)

def cure_cat_model(train_df, valid_df, test_df, cols):
    Xtr, Xva, Xte = prepare_xy(train_df, valid_df, test_df, cols)
    pred_va = np.zeros((len(valid_df), 4), dtype=float)
    pred_te = np.zeros((len(test_df), 4), dtype=float)

    y_event = train_df["event"].to_numpy(dtype=int)
    if len(np.unique(y_event)) < 2:
        base = y_event.mean()
        pred_va[:] = base
        pred_te[:] = base
        pred_va[:, -1] = 1.0
        pred_te[:, -1] = 1.0
        return monotone_project(pred_va), monotone_project(pred_te)

    event_model = fit_cat_classifier(Xtr, y_event, seed=SEED + 111, sample_weight=class_balanced_weights(y_event))
    p_event_va = predict_model(event_model, Xva)
    p_event_te = predict_model(event_model, Xte)

    hit_mask = train_df["event"].to_numpy(dtype=int) == 1
    if hit_mask.sum() >= 12:
        log_t = np.log(np.maximum(train_df.loc[hit_mask, "time_to_hit_hours"].to_numpy(dtype=float), 0.25))
        time_model = fit_cat_regressor(Xtr.loc[hit_mask], log_t, seed=SEED + 222)
        mu_tr = time_model.predict(Xtr.loc[hit_mask])
        sigma = float(np.std(log_t - mu_tr))
        sigma = float(np.clip(sigma, 0.35, 1.25))
        mu_va = time_model.predict(Xva)
        mu_te = time_model.predict(Xte)
        for j, H in enumerate(HORIZONS):
            if H == 72:
                pred_va[:, j] = 1.0
                pred_te[:, j] = 1.0
            else:
                z_va = (np.log(max(H, 0.25)) - mu_va) / sigma
                z_te = (np.log(max(H, 0.25)) - mu_te) / sigma
                pred_va[:, j] = p_event_va * norm.cdf(z_va)
                pred_te[:, j] = p_event_te * norm.cdf(z_te)
    else:
        for j, H in enumerate(HORIZONS):
            if H == 72:
                pred_va[:, j] = 1.0
                pred_te[:, j] = 1.0
            else:
                pred_va[:, j] = p_event_va * (1.0 if H >= 48 else 0.75)
                pred_te[:, j] = p_event_te * (1.0 if H >= 48 else 0.75)

    return monotone_project(pred_va), monotone_project(pred_te)

def hazard_global_model(train_df, valid_df, test_df, cols, kind="lgb", seeds=(SEED, SEED + 1, SEED + 2)):
    Xtr, Xva, Xte = prepare_xy(train_df, valid_df, test_df, cols)
    hazards_va = np.zeros((len(valid_df), 4), dtype=float)
    hazards_te = np.zeros((len(test_df), 4), dtype=float)

    for j, (start, end) in enumerate(INTERVALS):
        if end == 72:
            hazards_va[:, j] = 1.0
            hazards_te[:, j] = 1.0
            continue
        y, mask = interval_label(train_df, start, end)
        if mask.sum() < 10 or len(np.unique(y[mask])) < 2:
            base = y[mask].mean() if mask.sum() else 0.01
            hazards_va[:, j] = base
            hazards_te[:, j] = base
            continue

        preds_va_seed = []
        preds_te_seed = []
        for seed in seeds:
            sw = class_balanced_weights(y[mask])
            if kind == "lgb":
                model = fit_lgb_classifier(Xtr.loc[mask], y[mask], seed=seed, sample_weight=sw)
            else:
                model = fit_cat_classifier(Xtr.loc[mask], y[mask], seed=seed, sample_weight=sw)
            if model is None:
                continue
            preds_va_seed.append(predict_model(model, Xva))
            preds_te_seed.append(predict_model(model, Xte))
        if len(preds_va_seed) == 0:
            base = y[mask].mean()
            hazards_va[:, j] = base
            hazards_te[:, j] = base
        else:
            hazards_va[:, j] = np.mean(np.column_stack(preds_va_seed), axis=1)
            hazards_te[:, j] = np.mean(np.column_stack(preds_te_seed), axis=1)

    return monotone_project(hazards_to_cumulative(hazards_va)), monotone_project(hazards_to_cumulative(hazards_te))

def hazard_regime_lgb(train_df, valid_df, test_df, cols):
    Xtr, Xva, Xte = prepare_xy(train_df, valid_df, test_df, cols)
    ztr = zone_codes(train_df)
    zva = zone_codes(valid_df)
    zte = zone_codes(test_df)

    hazards_va = np.zeros((len(valid_df), 4), dtype=float)
    hazards_te = np.zeros((len(test_df), 4), dtype=float)

    for j, (start, end) in enumerate(INTERVALS):
        if end == 72:
            hazards_va[:, j] = 1.0
            hazards_te[:, j] = 1.0
            continue

        y, mask = interval_label(train_df, start, end)
        global_model = None
        if mask.sum() >= 10 and len(np.unique(y[mask])) >= 2:
            global_model = fit_lgb_classifier(Xtr.loc[mask], y[mask], seed=SEED + 500 + j, sample_weight=class_balanced_weights(y[mask]))
            global_va = predict_model(global_model, Xva)
            global_te = predict_model(global_model, Xte)
        else:
            base = y[mask].mean() if mask.sum() else 0.01
            global_va = np.full(len(valid_df), base)
            global_te = np.full(len(test_df), base)

        hazards_va[:, j] = global_va
        hazards_te[:, j] = global_te

        for z in np.unique(ztr):
            idx = mask & (ztr == z)
            pos = int(y[idx].sum())
            neg = int(idx.sum() - pos)
            if idx.sum() < 18 or pos < 3 or neg < 3:
                continue
            model = fit_lgb_classifier(Xtr.loc[idx], y[idx], seed=SEED + 800 + 17 * j + int(z), sample_weight=class_balanced_weights(y[idx]))
            if model is None:
                continue
            mva = zva == z
            mte = zte == z
            if mva.any():
                hazards_va[mva, j] = 0.30 * hazards_va[mva, j] + 0.70 * predict_model(model, Xva.loc[mva])
            if mte.any():
                hazards_te[mte, j] = 0.30 * hazards_te[mte, j] + 0.70 * predict_model(model, Xte.loc[mte])

    return monotone_project(hazards_to_cumulative(hazards_va)), monotone_project(hazards_to_cumulative(hazards_te))

def knn_local_model(train_df, valid_df, test_df, local_cols, k=25):
    local_cols = [c for c in local_cols if c in train_df.columns and c in valid_df.columns and c in test_df.columns]
    if len(local_cols) == 0:
        zeros_va = np.full((len(valid_df), 4), train_df["event"].mean(), dtype=float)
        zeros_te = np.full((len(test_df), 4), train_df["event"].mean(), dtype=float)
        zeros_va[:, -1] = 1.0
        zeros_te[:, -1] = 1.0
        return monotone_project(zeros_va), monotone_project(zeros_te)

    med = train_df[local_cols].median(numeric_only=True)
    Xtr = train_df[local_cols].fillna(med).to_numpy(dtype=float)
    Xva = valid_df[local_cols].fillna(med).to_numpy(dtype=float)
    Xte = test_df[local_cols].fillna(med).to_numpy(dtype=float)

    scaler = RobustScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xva = scaler.transform(Xva)
    Xte = scaler.transform(Xte)

    nn = NearestNeighbors(n_neighbors=min(k, len(train_df)), metric="euclidean")
    nn.fit(Xtr)

    d_va, i_va = nn.kneighbors(Xva)
    d_te, i_te = nn.kneighbors(Xte)

    pred_va = np.zeros((len(valid_df), 4), dtype=float)
    pred_te = np.zeros((len(test_df), 4), dtype=float)

    for j, H in enumerate(HORIZONS):
        if H == 72:
            pred_va[:, j] = 1.0
            pred_te[:, j] = 1.0
            continue
        y, mask = cumulative_label(train_df, H)
        global_mean = y[mask].mean() if mask.sum() else train_df["event"].mean()

        def local_predict(dmat, imat):
            out = np.zeros(len(imat), dtype=float)
            for r in range(len(imat)):
                idx = imat[r]
                ok = mask[idx]
                if ok.sum() < 3:
                    out[r] = global_mean
                    continue
                w = 1.0 / (0.05 + dmat[r][ok])
                yy = y[idx][ok]
                out[r] = np.average(yy, weights=w)
            return out

        pred_va[:, j] = local_predict(d_va, i_va)
        pred_te[:, j] = local_predict(d_te, i_te)

    return monotone_project(pred_va), monotone_project(pred_te)

def build_meta_frame(df, base_preds, interval_idx):
    feats = {}
    for name, p in base_preds.items():
        h = cumulative_to_hazards(p)
        feats[f"{name}_h"] = h[:, interval_idx]
        feats[f"{name}_p"] = p[:, interval_idx]
        if interval_idx > 0:
            feats[f"{name}_prevp"] = p[:, interval_idx - 1]
    for c in core_meta_cols:
        feats[c] = df[c].to_numpy(dtype=float)
    return pd.DataFrame(feats, index=df.index)

def fit_meta_logit(X_train, y_train, X_pred):
    if len(np.unique(y_train)) < 2 or len(y_train) < 12:
        base = float(np.mean(y_train)) if len(y_train) else 0.01
        return np.full(len(X_pred), base)
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=0.45, class_weight="balanced", max_iter=3000, random_state=SEED + 333))
    ])
    sw = class_balanced_weights(y_train)
    pipe.fit(X_train, y_train, clf__sample_weight=sw)
    return pipe.predict_proba(X_pred)[:, 1]

def search_blend_weights(train_df, candidates):
    names = list(candidates.keys())
    mats = [candidates[n] for n in names]
    best = (-1e9, None, None)
    grid = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
    for wa in grid:
        for wb in grid:
            for wc in grid:
                if wa + wb + wc > 1:
                    continue
                wd = 1.0 - wa - wb - wc
                w = np.array([wa, wb, wc, wd])
                if (w < 0).any():
                    continue
                blend = sum(wi * mi for wi, mi in zip(w, mats))
                blend = monotone_project(blend)
                hybrid, c, wbrier, _ = hybrid_score_from_probs(train_df, blend)
                if hybrid > best[0]:
                    best = (hybrid, w, blend)
    return names, best[1], best[2], best[0]

strat = make_stratify_label(train)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(train, strat))

base_names = ["anchor", "linear", "cure", "lgb_global", "cat_global", "lgb_regime", "knn_local"]
oof_base = {name: np.zeros((len(train), 4), dtype=float) for name in base_names}
test_base_folds = {name: [] for name in base_names}

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    tr_df = train.iloc[tr_idx].reset_index(drop=True)
    va_df = train.iloc[va_idx].reset_index(drop=True)

    pred_va_anchor = empirical_anchor(tr_df, va_df)
    pred_te_anchor = empirical_anchor(tr_df, test)
    oof_base["anchor"][va_idx] = pred_va_anchor
    test_base_folds["anchor"].append(pred_te_anchor)

    pred_va_linear, pred_te_linear = linear_cumulative_model(tr_df, va_df, test, feature_cols)
    oof_base["linear"][va_idx] = pred_va_linear
    test_base_folds["linear"].append(pred_te_linear)

    pred_va_cure, pred_te_cure = cure_cat_model(tr_df, va_df, test, feature_cols)
    oof_base["cure"][va_idx] = pred_va_cure
    test_base_folds["cure"].append(pred_te_cure)

    pred_va_lgb, pred_te_lgb = hazard_global_model(tr_df, va_df, test, feature_cols, kind="lgb")
    oof_base["lgb_global"][va_idx] = pred_va_lgb
    test_base_folds["lgb_global"].append(pred_te_lgb)

    pred_va_cat, pred_te_cat = hazard_global_model(tr_df, va_df, test, feature_cols, kind="cat", seeds=(SEED + 10, SEED + 11))
    oof_base["cat_global"][va_idx] = pred_va_cat
    test_base_folds["cat_global"].append(pred_te_cat)

    pred_va_reg, pred_te_reg = hazard_regime_lgb(tr_df, va_df, test, feature_cols)
    oof_base["lgb_regime"][va_idx] = pred_va_reg
    test_base_folds["lgb_regime"].append(pred_te_reg)

    local_cols = [c for c in [
        "dist_km","log_dist","closing_pos","along_pos","alignment_abs","advance_ratio","threat_proxy",
        "growth_rate_log","radial_rate_log","size_dist_ratio","quality_score"
    ] if c in feature_cols]
    pred_va_knn, pred_te_knn = knn_local_model(tr_df, va_df, test, local_cols)
    oof_base["knn_local"][va_idx] = pred_va_knn
    test_base_folds["knn_local"].append(pred_te_knn)

    print(f"fold {fold} done")

test_base = {name: np.mean(np.stack(mats, axis=0), axis=0) for name, mats in test_base_folds.items()}

oof_meta = np.zeros((len(train), 4), dtype=float)
test_meta_h = np.zeros((len(test), 4), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    tr_df = train.iloc[tr_idx].reset_index(drop=True)
    va_df = train.iloc[va_idx].reset_index(drop=True)

    base_tr = {name: oof_base[name][tr_idx] for name in base_names}
    base_va = {name: oof_base[name][va_idx] for name in base_names}

    haz_va = np.zeros((len(va_df), 4), dtype=float)

    for j, (start, end) in enumerate(INTERVALS):
        if end == 72:
            haz_va[:, j] = 1.0
            continue
        y_int, mask_int = interval_label(tr_df, start, end)
        X_meta_tr = build_meta_frame(tr_df, base_tr, j)
        X_meta_va = build_meta_frame(va_df, base_va, j)
        if mask_int.sum() < 12:
            haz_va[:, j] = np.mean(cumulative_to_hazards(base_va["lgb_global"])[:, j])
        else:
            haz_va[:, j] = fit_meta_logit(X_meta_tr.loc[mask_int], y_int[mask_int], X_meta_va)

    oof_meta[va_idx] = monotone_project(hazards_to_cumulative(haz_va))
    print(f"meta fold {fold} done")

for j, (start, end) in enumerate(INTERVALS):
    if end == 72:
        test_meta_h[:, j] = 1.0
        continue
    y_int, mask_int = interval_label(train, start, end)
    X_meta_tr = build_meta_frame(train, oof_base, j)
    X_meta_te = build_meta_frame(test, test_base, j)
    if mask_int.sum() < 12:
        test_meta_h[:, j] = np.mean(cumulative_to_hazards(test_base["lgb_global"])[:, j])
    else:
        test_meta_h[:, j] = fit_meta_logit(X_meta_tr.loc[mask_int], y_int[mask_int], X_meta_te)

test_meta = monotone_project(hazards_to_cumulative(test_meta_h))

candidates_oof = {
    "meta": oof_meta,
    "cure": oof_base["cure"],
    "anchor": oof_base["anchor"],
    "knn": oof_base["knn_local"],
}
names, weights, best_oof_blend, best_hybrid = search_blend_weights(train, candidates_oof)
print("blend components =", names)
print("blend weights =", weights)
print("best OOF blend hybrid =", best_hybrid)

candidates_test = {
    "meta": test_meta,
    "cure": test_base["cure"],
    "anchor": test_base["anchor"],
    "knn": test_base["knn_local"],
}
final_test = sum(w * candidates_test[n] for w, n in zip(weights, names))
final_oof = best_oof_blend.copy()

extra_paths = []
for p in glob.glob("/kaggle/input/**/*.csv", recursive=True):
    name = os.path.basename(p).lower()
    if "submission" in name or "samplecsv_sub" in name:
        if (DATA_DIR / "sample_submission.csv").exists():
            if Path(p).resolve() == (DATA_DIR / "sample_submission.csv").resolve():
                continue
        extra_paths.append(p)

external_candidates = []
for p in sorted(set(extra_paths)):
    try:
        df_ext = pd.read_csv(p)
        need = set(REQUIRED_SUB_COLS)
        if need.issubset(df_ext.columns) and len(df_ext) == len(test):
            df_ext = df_ext[REQUIRED_SUB_COLS].copy()
            df_ext = df_ext.sort_values("event_id").reset_index(drop=True)
            test_ids_sorted = test[["event_id"]].sort_values("event_id").reset_index(drop=True)
            if df_ext["event_id"].tolist() == test_ids_sorted["event_id"].tolist():
                ext_pred = df_ext[[c for c in REQUIRED_SUB_COLS if c != "event_id"]].to_numpy(dtype=float)
                external_candidates.append(ext_pred)
    except Exception:
        pass

if len(external_candidates) > 0:
    ext_consensus = np.median(np.stack(external_candidates, axis=0), axis=0)
    final_test_sorted = pd.DataFrame({
        "event_id": test["event_id"].to_numpy(),
        "prob_12h": final_test[:, 0],
        "prob_24h": final_test[:, 1],
        "prob_48h": final_test[:, 2],
        "prob_72h": final_test[:, 3],
    }).sort_values("event_id").reset_index(drop=True)
    final_test_sorted.iloc[:, 1:] = 0.90 * final_test_sorted.iloc[:, 1:].to_numpy() + 0.10 * ext_consensus
    final_test = final_test_sorted.set_index("event_id").loc[test["event_id"]].to_numpy()
    print("External CSV candidates found:", len(external_candidates))
else:
    print("No external CSV candidates found.")

final_oof = monotone_project(final_oof)
final_test = monotone_project(final_test)
final_oof[:, 3] = 1.0
final_test[:, 3] = 1.0
final_oof = monotone_project(final_oof)
final_test = monotone_project(final_test)

hybrid, c, wb, briers = hybrid_score_from_probs(train, final_oof)
print(f"OOF Hybrid = {hybrid:.6f}")
print(f"OOF C-Index = {c:.6f}")
print(f"OOF WBrier = {wb:.6f}")
for H in [12, 24, 48, 72]:
    print(f"OOF Brier@{H} = {briers[H]:.6f}")

submission = pd.DataFrame({
    "event_id": test["event_id"].astype(sample_submission["event_id"].dtype if sample_submission is not None else test["event_id"].dtype),
    "prob_12h": final_test[:, 0],
    "prob_24h": final_test[:, 1],
    "prob_48h": final_test[:, 2],
    "prob_72h": final_test[:, 3],
})

submission = submission[REQUIRED_SUB_COLS].copy()
submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = np.clip(
    monotone_project(submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].to_numpy()),
    0.0, 1.0
)
submission.to_csv("/kaggle/working/submission.csv", index=False)

assert list(submission.columns) == REQUIRED_SUB_COLS
assert submission["event_id"].is_unique
assert len(submission) == len(test)
assert set(submission["event_id"]) == set(test["event_id"])
assert ((submission.iloc[:, 1:] >= 0).all().all() and (submission.iloc[:, 1:] <= 1).all().all())
assert (submission["prob_12h"] <= submission["prob_24h"]).all()
assert (submission["prob_24h"] <= submission["prob_48h"]).all()
assert (submission["prob_48h"] <= submission["prob_72h"]).all()

print("Saved: /kaggle/working/submission.csv")
print(submission.head())


DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN = (221, 81) TEST = (95, 79) N_FEATURES = 78
fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done
meta fold 1 done
meta fold 2 done
meta fold 3 done
meta fold 4 done
meta fold 5 done
blend components = ['meta', 'cure', 'anchor', 'knn']
blend weights = [0.15 0.8  0.   0.05]
best OOF blend hybrid = 0.9721355902653657
No external CSV candidates found.
OOF Hybrid = 0.972136
OOF C-Index = 0.941703
OOF WBrier = 0.014822
OOF Brier@12 = 0.056443
OOF Brier@24 = 0.025685
OOF Brier@48 = 0.017791
OOF Brier@72 = 0.000000
Saved: /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.000764  0.001220  0.001310       1.0
1  13353600  0.856548  0.980304  0.992232       1.0
2  13942327  0.001867  0.002461  0.002607       1.0
3  16112781  0.928445  0.986778  0.994287       1.0
4  17132808  0.036058  0.039214  0.043938       1.0
